<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 00. LeNet-5 — Gradient-Based Learning Applied to Document Recognition

- **원 논문:** [Gradient-Based Learning Applied to Document Recognition](http://yann.lecun.com/exdb/publis/pdf/lecun-98.pdf)
- **저자 / 발표:** Yann LeCun, Léon Bottou, Yoshua Bengio, and Patrick Haffner · Proceedings of the IEEE 86(11) (1998)
- **난이도 / 예상 시간:** 입문 · 약 70분
- **선수 지식:** PyTorch tensor, Conv2d, pooling, cross-entropy의 기초

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

32×32 입력에서 C1-S2-C3-S4-C5-F6으로 이어지는 LeNet-5의 공간 크기 변화를 재현하고, 공유 합성곱 필터와 평균 subsampling을 사용한 작은 분류기를 합성 패턴에 학습한다. 각 층의 shape, 유한한 loss, 한 번 이상의 loss 감소를 검증한다.

**원 논문과 다른 것**

원 논문은 MNIST 및 문서 인식 파이프라인, 부분 연결 C3, 학습 가능한 subsampling 계수, 84차원 F6과 Euclidean RBF 출력까지 사용한다. 여기서는 다운로드 없는 20개 합성 영상, 완전 연결 C3, AvgPool2d, 10-way linear logits로 핵심 계층 구조만 재현한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §II.B ‘LeNet-5’와 Fig. 2: 32×32 → C1(6@28²) → S2(6@14²) | `LeNet5Mini.c1`, `s2`와 shape 추적 셀 | assert가 `(6,28,28)`과 `(6,14,14)`를 직접 확인 |
| §II.B, Fig. 2: C3(16@10²) → S4(16@5²) → C5(120@1²) | `c3`, `s4`, `c5` 및 중간 활성 시각화 | 5×5 valid convolution과 2×2 평균 subsampling의 출력 크기 검증 |
| §II.B ‘LeNet-5’: tanh 비선형성과 F6의 84 units | `torch.tanh`와 `f6 = nn.Linear(120, 84)` | 파라미터/출력 shape와 10-class logits 검증 |
| §II.A ‘Convolutional Networks’: local receptive fields와 shared weights | C1 feature-map 시각화와 합성 위치 패턴 학습 | 동일 Conv2d kernel이 전체 영상 위치에 적용되는 결과를 관찰 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
h^{(l)}_{k,i,j}=\sigma\!\left(
b_k^{(l)}+\sum_c\sum_{u,v}W^{(l)}_{k,c,u,v}
h^{(l-1)}_{c,i+u,j+v}\right)
$$

- $h^{(l)}$은 $l$번째 feature map, $W$는 공유되는 convolution kernel입니다.
- 핵심 역할은 국소 receptive field와 weight sharing으로 공간 패턴을 추출하는 것입니다.
- 코드의 convolution/pooling 모듈이 식의 합과 샘플링을 담당합니다.
- shape은 $[B,C,H,W]\rightarrow[B,C',H',W']$로 변하고 classifier에서 $[B,K]$가 됩니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §II.B ‘LeNet-5’와 Fig. 2: 32×32 → C1(6@28²) → S2(6@14²)
- **노트북 구현:** `LeNet5Mini.c1`, `s2`와 shape 추적 셀
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** assert가 `(6,28,28)`과 `(6,14,14)`를 직접 확인를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 MNIST 및 문서 인식 파이프라인, 부분 연결 C3, 학습 가능한 subsampling 계수, 84차원 F6과 Euclidean RBF 출력까지 사용한다. 여기서는 다운로드 없는 20개 합성 영상, 완전 연결 C3, AvgPool2d, 10-way linear logits로 핵심 계층 구조만 재현한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. 논문 구조를 읽는 법

Fig. 2에서 `Cx`는 학습되는 합성곱 층, `Sx`는 해상도를 절반으로 줄이는
subsampling 층입니다. 원 논문의 S2/S4는 2×2 평균에 학습 가능한 계수와 bias를
붙였지만, 이 미니 재현에서는 평균 연산 자체에 초점을 맞춰 `AvgPool2d`를 씁니다.
또한 원 C3의 **부분 연결표**는 완전 연결 합성곱으로 단순화합니다. 그러므로 여기서
확인할 것은 최종 MNIST 정확도가 아니라 Fig. 2의 receptive-field/shape 흐름입니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image [B,1,32,32] → feature [B,C,H,W] → logits [B,10]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(0)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def make_position_patterns(repeats=2, size=32):
    # 10개 위치를 서로 다른 class로 쓰는 다운로드 없는 toy digits.
    generator = torch.Generator().manual_seed(101)
    images, labels = [], []
    for repeat in range(repeats):
        for label in range(10):
            image = torch.zeros(1, size, size)
            row = 4 + (label // 5) * 14
            col = 2 + (label % 5) * 6
            image[:, row : row + 7, col : col + 5] = 1.0
            # class마다 작은 가로 획을 더해 단순 위치 복사 이상의 패턴을 만든다.
            image[:, row + label % 4 : row + label % 4 + 2, col : col + 8] = 0.55
            noise = 0.04 * torch.randn(image.shape, generator=generator)
            images.append((image + noise).clamp(0, 1))
            labels.append(label)
    return torch.stack(images), torch.tensor(labels)


lenet_x, lenet_y = make_position_patterns()
lenet_x, lenet_y = ACCELERATOR.move(lenet_x, lenet_y)
assert lenet_x.shape == (20, 1, 32, 32)
assert lenet_y.tolist() == list(range(10)) * 2
print(ACCELERATOR.summary())
print("synthetic batch:", tuple(lenet_x.shape))

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §II.B ‘LeNet-5’와 Fig. 2: 32×32 → C1(6@28²) → S2(6@14²) / §II.B, Fig. 2: C3(16@10²) → S4(16@5²) → C5(120@1²) / §II.B ‘LeNet-5’: tanh 비선형성과 F6의 84 units
- **이 셀의 책임:** `LeNet5Mini.c1`, `s2`와 shape 추적 셀 / `c3`, `s4`, `c5` 및 중간 활성 시각화 / `torch.tanh`와 `f6 = nn.Linear(120, 84)`
- **Tensor shape 계약:** image [B,1,32,32] → feature [B,C,H,W] → logits [B,10]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** assert가 `(6,28,28)`과 `(6,14,14)`를 직접 확인 / 5×5 valid convolution과 2×2 평균 subsampling의 출력 크기 검증 / 파라미터/출력 shape와 10-class logits 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: Fig. 2를 따라 C1-S2-C3-S4-C5-F6을 정의하세요.
class LeNet5Mini(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


lenet = LeNet5Mini().to(DEVICE)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 평가·완료 증거

- **원문 위치:** §II.B ‘LeNet-5’와 Fig. 2: 32×32 → C1(6@28²) → S2(6@14²) / §II.B, Fig. 2: C3(16@10²) → S4(16@5²) → C5(120@1²) / §II.B ‘LeNet-5’: tanh 비선형성과 F6의 84 units
- **이 셀의 책임:** `LeNet5Mini.c1`, `s2`와 shape 추적 셀 / `c3`, `s4`, `c5` 및 중간 활성 시각화 / `torch.tanh`와 `f6 = nn.Linear(120, 84)`
- **Tensor shape 계약:** image [B,1,32,32] → feature [B,C,H,W] → logits [B,10]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** assert가 `(6,28,28)`과 `(6,14,14)`를 직접 확인 / 5×5 valid convolution과 2×2 평균 subsampling의 출력 크기 검증 / 파라미터/출력 shape와 10-class logits 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 중간 텐서를 계산하고 Fig. 2의 다섯 공간 크기를 assert로 검증하세요.
def trace_lenet_features(model, batch):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 4. 원 논문과 이 구현의 경계

C3의 모든 출력 map을 모든 S2 map에 연결한 것은 원 논문의 연결표와 다릅니다.
출력도 원 논문의 RBF distance가 아니라 오늘날 익숙한 linear logits입니다. 반면
32×32 입력, 5×5 valid convolution, 2×2 평균 subsampling, `6→16→120→84`라는
핵심 shape 경로는 그대로이므로 합성곱 계층의 설계 아이디어를 검증할 수 있습니다.

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §II.A ‘Convolutional Networks’: local receptive fields와 shared weights
- **이 셀의 책임:** C1 feature-map 시각화와 합성 위치 패턴 학습
- **Tensor shape 계약:** image [B,1,32,32] → feature [B,C,H,W] → logits [B,10]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 동일 Conv2d kernel이 전체 영상 위치에 적용되는 결과를 관찰. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: Adam과 cross_entropy로 15 step 학습하세요.
def train_lenet_step(model, batch, targets, optimizer):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.